In [0]:
from pyspark.sql.functions import *

df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/default/pyspark_files/BigMart Sales.csv"))

display(df)

In [0]:
df.limit(5).display()

## Aggregation and Analytics

Aggregation involves grouping data to calculate summary statistics.

* **groupBy** **:** Categorises data.
* **agg** **:** Used after `groupBy` to perform multiple calculations like `sum`, `avg`, `max`, or `min`.
* **collect\_list** **:** Instead of summing, it aggregates all values in a group into a single list (similar to `group_concat` in SQL).
* **pivot** **:** Rotates data from rows into columns for reporting (Excel-style).

In [0]:
df.groupby("Item_Type").agg(sum("Item_MRP")).display() #agg() -> sum(), avg(), count(), max(), etc

In [0]:
df.groupby("Item_Type", "Outlet_Size").agg(sum("Item_MRP").alias("Total_MRP")).display()

In [0]:
df.groupby("Item_Type", "Outlet_Size").agg(sum("Item_MRP").alias("Total_MRP"), avg("Item_MRP").alias("Avg_MRP")).display()

###Collect_List

In [0]:
from pyspark.sql.functions import *

In [0]:
data = [('user1', 'book1'), ('user1', 'book2'), ('user2', 'book2'), ('user2', 'book4'), ('user3', 'book1')]
schema = 'user string, book string'

df_book = spark.createDataFrame(data, schema)
df_book.display()

In [0]:
df_book.groupby('user').agg(collect_list("book")).display()

### Pivot

In [0]:
df.select('Item_Type','Outlet_Size','Item_MRP').display()

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

##  Conditional Logic: When/Otherwise

The `when()` and `otherwise()` functions replicate `IF-THEN-ELSE` or SQL `CASE WHEN` logic.

* Example: `withColumn("Flag", when(col("price") > 100, "Expensive").otherwise("Cheap"))`.

In [0]:
df_short = df.select('Item_Identifier','Item_Type', 'Item_MRP')
df_short.display()

In [0]:
df_short = df_short.withColumn('veg_flag', when(col('Item_Type') == 'Meat', "Non veg").otherwise('Veg'))
df_short.display()

In [0]:
df_short = df_short.withColumn("veg_exp_flag", when((col('veg_flag') == 'Veg') & (col('Item_MRP') > 100), "Veg_Expensive")\
                .when((col('veg_flag') == 'Veg') & (col('Item_MRP') < 100), "Veg_Inexpensive")\
                .when((col('veg_flag') == 'Non veg') & (col('Item_MRP') > 100), "NonVeg_Expensive")\
                .otherwise("NonVeg_Inexpensive")
               ).display()

## Joins

Combining DataFrames requires a joining key.

| **Join Type** | **Description**                                                                                                 |
| ------------- | --------------------------------------------------------------------------------------------------------------- |
| **Inner**     | Returns only records with matching keys in both DataFrames.                                                     |
| **Left**      | Returns all records from the left, plus matches from the right (nulls where no match).                          |
| **Right**     | Returns all records from the right, plus matches from the left.                                                 |
| **Anti**      | **Critical Interview Topic:** Returns only records from the left DataFrame that have **no match** in the right. |

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id']).display() #by default its inner so we dont need to write 

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'left').display() #left join

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'right').display() #right join

####**Anti Join** -> Returns only records from the left DataFrame that have **no match** in the right. 

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'anti').display() #